# Disease Similarity Scoring on Gene and Disease Networks Using node2vec
### Project-1 — Human Genes and Gene Relationships with Diseases

**Goal:** given two diseases D1 and D2, return a similarity score in **[0, 1]**
(0 = unrelated, 0.2 = not really similar, 0.6 = somewhat similar, 1 = maximally similar).

**Uses all three CTD downloads:** #7 curated gene-disease (required), #18 gene vocabulary
(UniProt for BLAST), #16 disease vocabulary (MeSH hierarchy for C4 validation).

**Methodology — three classes:**
- **A. Data Processing** — A1 collect inputs (incl. NCBI fetch), A2 parse & integrate, A3 resolve D1/D2, A4 gene-significance weight, A5 sequence/BLAST similarity (real blastp or k-mer), A6 gene ID mapping (NCBI · OMIM · UniProt)
- **B. Data Modelling** — B1 build graph, B2 biased random walks (node2vec), B3 skip-gram embedding
- **C. Data Evaluation** — C1 baselines, C2 similarity scoring, C3 calibration & banding, C4 validation vs MeSH hierarchy (#16)
- **Visualizations** — shared-gene network, method comparison, embedding map, calibration

Run the cells top to bottom.

In [ ]:
# --- setup ---
%matplotlib inline
import os, csv, io, math, random, time, json, shutil, subprocess, tempfile
from collections import defaultdict
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
import matplotlib.pyplot as plt
import pandas as pd

HERE   = os.getcwd()
INPUTS = os.path.join(HERE, "inputs")
DEF_CTD      = os.path.join(INPUTS, "CTD_curated_genes_diseases.csv")   # #7  (required)
DEF_OMIM     = os.path.join(INPUTS, "omim_genemap.tsv")
DEF_FASTA    = os.path.join(INPUTS, "ncbi_gene_proteins.fasta")
DEF_GENES    = os.path.join(INPUTS, "CTD_genes.csv")                    # #18 UniProt (A6)
DEF_OMIMGENE = os.path.join(INPUTS, "omim_gene_numbers.tsv")
DEF_DISVOCAB = os.path.join(INPUTS, "CTD_diseases.csv")                 # #16 MeSH (C4)
print("inputs folder:", INPUTS)

## Step A — Data Processing

In [ ]:
# ---- A4  Gene significance weight ----
def gene_significance_weight(pubmed_ids, evidence):
    n_pmid = len([p for p in pubmed_ids.split("|") if p.strip()])
    w = 1.0 + math.log1p(n_pmid)
    if "marker/mechanism" in evidence: w += 0.5
    return w

# ---- A1 + A2  CTD (#7) -> disease->gene map ----
def load_ctd(path):
    dis_genes = defaultdict(dict); name2id = defaultdict(set); id2name = {}
    omim2mesh = defaultdict(set);  gene_deg = defaultdict(int); gene2id = {}
    rows = [l for l in open(path, encoding="utf-8") if not l.startswith("#")]
    for r in csv.reader(io.StringIO("".join(rows))):
        if len(r) < 7: continue
        gs, gid, dn, did, ev, omim, pmids = r[:7]
        dis_genes[did][gs] = max(dis_genes[did].get(gs, 0.0), gene_significance_weight(pmids, ev))
        gene2id.setdefault(gs, gid)
        id2name[did] = dn; name2id[dn.lower()].add(did)
        for o in omim.split("|"):
            if o.strip(): omim2mesh[o.strip()].add(did)
    for did, genes in dis_genes.items():
        for g in genes: gene_deg[g] += 1
    return dict(dis_genes), name2id, id2name, omim2mesh, gene_deg, gene2id

In [ ]:
# ---- A1+A2  OMIM (#2), NCBI FASTA (#1), and Disease vocabulary (#16 -> MeSH) ----
def load_omim(path):
    omim_genes = defaultdict(dict); omim_name = {}
    if not os.path.exists(path): return {}, {}
    for line in open(path, encoding="utf-8"):
        if line.startswith("#") or line.startswith("PhenotypeMIM"): continue
        p = line.rstrip("\n").split("\t")
        if len(p) < 4: continue
        mim, pname, gs, gid = (x.strip() for x in p[:4])
        omim_genes["OMIM:"+mim][gs] = 1.5; omim_name["OMIM:"+mim] = pname
    return dict(omim_genes), omim_name

def load_fasta(path):
    seqs, sym, buf = {}, None, []
    if not os.path.exists(path): return seqs
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        if not line or line.startswith(";"): continue
        if line.startswith(">"):
            if sym: seqs[sym] = "".join(buf)
            sym = line[1:].split("|")[0].strip(); buf = []
        else: buf.append(line.strip())
    if sym: seqs[sym] = "".join(buf)
    return seqs

def load_disease_vocab(path):   # #16 MEDIC -> {DiseaseID: set of MeSH categories}
    did2tree = {}
    if not os.path.exists(path): return did2tree
    csv.field_size_limit(10_000_000)
    with open(path, encoding="utf-8") as fh:
        for r in csv.reader(l for l in fh if not l.startswith("#")):
            if len(r) < 6 or r[1] == "DiseaseID": continue
            cats = {t.split("/")[0].split(".")[0] for t in r[5].split("|") if t.strip()}
            if cats: did2tree[r[1]] = cats
    return did2tree

In [ ]:
# ---- A5  Sequence / BLAST similarity ----
# (a) real blastp (needs NCBI BLAST+ on PATH)   (b) k-mer cosine fallback
def kmer_vec(seq, k=3):
    v = defaultdict(float)
    for i in range(len(seq)-k+1): v[seq[i:i+k]] += 1.0
    return v
def seq_similarity(a, b, k=3):
    va, vb = kmer_vec(a,k), kmer_vec(b,k)
    if not va or not vb: return 0.0
    dot = sum(va[x]*vb.get(x,0.0) for x in va)
    na = math.sqrt(sum(x*x for x in va.values())); nb = math.sqrt(sum(x*x for x in vb.values()))
    return dot/(na*nb) if na and nb else 0.0

def blast_available():
    return bool(shutil.which("makeblastdb") and shutil.which("blastp"))

def blast_homology_edges(seqs, thr=0.30, evalue=10.0):
    """REAL all-vs-all blastp -> [(g1,g2,sim)] with sim = normalized bit-score in [0,1].
    Returns None if BLAST+ is not installed (caller falls back to k-mer)."""
    if not blast_available(): return None
    with tempfile.TemporaryDirectory() as td:
        fa = os.path.join(td,"genes.faa"); db = os.path.join(td,"db")
        with open(fa,"w") as f:
            for g,s in seqs.items():
                if s: f.write(f">{g}\n{s}\n")
        subprocess.run(["makeblastdb","-in",fa,"-dbtype","prot","-out",db], check=True, capture_output=True)
        res = subprocess.run(["blastp","-query",fa,"-db",db,"-evalue",str(evalue),
                              "-outfmt","6 qseqid sseqid pident bitscore"],
                             check=True, capture_output=True, text=True).stdout
    self_bit, pairs = {}, {}
    for line in res.splitlines():
        q,s,pid,bit = line.split("\t"); bit=float(bit)
        if q==s: self_bit[q] = max(self_bit.get(q,0.0), bit)
    for line in res.splitlines():
        q,s,pid,bit = line.split("\t")
        if q>=s: continue
        norm = float(bit)/max(self_bit.get(q,1.0), self_bit.get(s,1.0), 1.0)
        pairs[(q,s)] = max(pairs.get((q,s),0.0), min(norm,1.0))
    return [(a,b,v) for (a,b),v in pairs.items() if v>=thr]

In [ ]:
# ---- A3  Resolve query -> node ;  A6  gene ID mapping ----
def resolve(query, name2id, id2name, omim2mesh, omim_name, dis_genes, G):
    q = query.strip()
    if q.startswith("OMIM:") or q.isdigit():
        mim = q.split(":")[-1]
        if mim in omim2mesh:
            best = max(omim2mesh[mim], key=lambda d: len(dis_genes.get(d,{}))); return best, id2name.get(best,best)
        if ("OMIM:"+mim) in G: return "OMIM:"+mim, omim_name.get("OMIM:"+mim,"OMIM:"+mim)
    if q in G: return q, id2name.get(q,q)
    ql = q.lower()
    if ql in name2id:
        best = max(name2id[ql], key=lambda d: len(dis_genes.get(d,{}))); return best, id2name[best]
    cand = [(d,id2name[d],len(dis_genes.get(d,{}))) for nm,ids in name2id.items() if ql in nm for d in ids]
    if cand:
        best = max(cand, key=lambda x: x[2]); return best[0], best[1]
    return None, None

def resolve_name_only(query, dis_genes, name2id, id2name):
    ql = query.strip().lower()
    cand = [d for nm,ids in name2id.items() if ql in nm for d in ids]
    return max(cand, key=lambda d: len(dis_genes.get(d,{}))) if cand else None

def load_gene_vocab(path):        # #18 -> UniProt
    vocab = {}
    if not os.path.exists(path): return vocab
    csv.field_size_limit(10_000_000)
    with open(path, encoding="utf-8") as fh:
        for r in csv.reader(fh):
            if not r or r[0].startswith("#") or len(r) < 8: continue
            vocab[r[0]] = {"geneid": r[2], "uniprot": r[7].split("|")[0], "name": r[1]}
    return vocab

def load_omim_gene_numbers(path):
    mp = {}
    if not os.path.exists(path): return mp
    for line in open(path, encoding="utf-8"):
        if line.startswith("#") or line.startswith("GeneSymbol"): continue
        p = line.rstrip("\n").split("\t")
        if len(p) >= 2 and p[0].strip(): mp[p[0].strip()] = p[1].strip()
    return mp

def gene_id_table(genes, gene2id, gene_vocab, omim_gene):
    rows = []
    for g in sorted(genes):
        ncbi = gene2id.get(g) or gene_vocab.get(g, {}).get("geneid", "")
        rows.append((g, ncbi, omim_gene.get(g,""), gene_vocab.get(g,{}).get("uniprot","")))
    return rows

### A1 (NCBI fetch) — real protein sequences for BLAST (needs internet)
Optional: run to replace the sample FASTA with real sequences.

In [ ]:
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
def _eutil(endpoint, params, api_key=None):
    import urllib.request, urllib.parse
    if api_key: params["api_key"] = api_key
    return urllib.request.urlopen(BASE+endpoint+"?"+urllib.parse.urlencode(params), timeout=30).read().decode()

def fetch_protein_fasta(symbol, api_key=None):
    term = f"{symbol}[gene] AND Homo sapiens[orgn] AND refseq[filter]"
    j = json.loads(_eutil("esearch.fcgi", {"db":"protein","term":term,"retmax":1,"retmode":"json"}, api_key))
    ids = j.get("esearchresult", {}).get("idlist", [])
    if not ids: return None
    fasta = _eutil("efetch.fcgi", {"db":"protein","id":ids[0],"rettype":"fasta","retmode":"text"}, api_key)
    return "".join(l.strip() for l in fasta.splitlines() if l and not l.startswith(">")) or None

def fetch_sequences_for_genes(gene_ncbi_map, out_path=DEF_FASTA, api_key=None, max_genes=60, delay=0.34):
    written = 0
    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write("; Real human protein sequences fetched from NCBI E-utilities\n")
        for i,(sym,gid) in enumerate(list(gene_ncbi_map.items())[:max_genes], 1):
            try: seq = fetch_protein_fasta(sym, api_key)
            except Exception as e: print(f"  [{i}] {sym}: ERROR {e}"); seq = None
            if seq: fh.write(f">{sym}|{gid}\n{seq}\n"); written += 1; print(f"  [{i}] {sym}: {len(seq)} aa")
            time.sleep(delay)
    print(f"Wrote {written} sequences -> {out_path}"); return written
print("NCBI fetch ready (call fetch_sequences_for_genes(...)).")

In [ ]:
# ---- RUN STEP A : load all inputs (#7, #2, #1, #18, #16) ----
dis_genes, name2id, id2name, omim2mesh, gene_deg, gene2id = load_ctd(DEF_CTD)
omim_genes, omim_name = load_omim(DEF_OMIM)
seqs        = load_fasta(DEF_FASTA)
gene_vocab  = load_gene_vocab(DEF_GENES)          # #18 UniProt (needs CTD_genes.csv)
omim_gene   = load_omim_gene_numbers(DEF_OMIMGENE)
did2tree    = load_disease_vocab(DEF_DISVOCAB)     # #16 MeSH categories (for C4)
N = len(dis_genes)
print(f"CTD {N} diseases | OMIM {len(omim_genes)} | NCBI {len(seqs)} seqs | UniProt {len(gene_vocab)} | MeSH-vocab {len(did2tree)}")

D1, D2 = "MESH:D001943", "MESH:D010190"           # Breast, Pancreatic Neoplasms
g1, g2 = dis_genes[D1], dis_genes[D2]
shared = sorted(set(g1) & set(g2))
print(f"{id2name[D1]}: {len(g1)} genes | {id2name[D2]}: {len(g2)} genes | shared: {len(shared)}")
pd.DataFrame(gene_id_table(shared, gene2id, gene_vocab, omim_gene),
             columns=["Gene","NCBI_Entrez_ID","OMIM_Gene_#","UniProt_for_BLAST"])

## Step B — Data Modelling

In [ ]:
# ---- B1 build graph (with real-BLAST option) ; B2 walks ; B3 skip-gram ----
def build_graph(dis_genes, omim_genes, seqs, homology_thr=0.30, use_blast=False):
    G = nx.Graph()
    def add_disease(node, genes):
        G.add_node(node, kind="disease")
        for gs, w in genes.items():
            G.add_node(gs, kind="gene")
            if G.has_edge(node, gs): G[node][gs]["weight"] = max(G[node][gs]["weight"], w)
            else: G.add_edge(node, gs, weight=w)
    for did, genes in dis_genes.items():  add_disease(did, genes)
    for did, genes in omim_genes.items(): add_disease(did, genes)
    present = {g: seqs[g] for g in seqs if g in G}
    homo = 0
    edges = blast_homology_edges(present, homology_thr) if use_blast else None
    if edges is not None:                                  # A5 real BLAST
        print("  A5: using REAL blastp")
        for a,b,s in edges: G.add_edge(a,b,weight=s); homo += 1
    else:
        if use_blast: print("  A5: BLAST+ not found -> k-mer fallback")
        gs = list(present)
        for i in range(len(gs)):
            for j in range(i+1, len(gs)):
                s = seq_similarity(present[gs[i]], present[gs[j]])
                if s >= homology_thr: G.add_edge(gs[i], gs[j], weight=s); homo += 1
    return G, homo

def precompute(G):
    nbrs, wts, nbrset = {}, {}, {}
    for n in G.nodes():
        ns = list(G.neighbors(n)); nbrs[n] = ns
        wts[n] = np.array([G[n][m]["weight"] for m in ns], dtype=float); nbrset[n] = set(ns)
    return nbrs, wts, nbrset

def one_walk(start, length, p, q, nbrs, wts, nbrset, rng):
    walk = [start]
    if not nbrs[start]: return walk
    walk.append(rng.choice(nbrs[start], p=wts[start]/wts[start].sum()))
    while len(walk) < length:
        cur, prev = walk[-1], walk[-2]; ns = nbrs[cur]
        if not ns: break
        w = wts[cur].copy()
        for idx, x in enumerate(ns):
            if x == prev: w[idx] /= p
            elif x not in nbrset[prev]: w[idx] /= q
        walk.append(rng.choice(ns, p=w/w.sum()))
    return [str(x) for x in walk]

def node2vec_embed(G, dim=128, walk_len=40, num_walks=10, p=1.0, q=0.5, window=5, seed=42, workers=4):
    nbrs, wts, nbrset = precompute(G); rng = np.random.default_rng(seed)
    nodes = list(G.nodes()); walks = []
    for _ in range(num_walks):
        rng.shuffle(nodes)
        for n in nodes: walks.append(one_walk(n, walk_len, p, q, nbrs, wts, nbrset, rng))
    return Word2Vec(walks, vector_size=dim, window=window, min_count=0, sg=1, workers=workers, seed=seed, epochs=5)

In [ ]:
# ---- RUN STEP B : build graph + train node2vec ----
USE_BLAST = False   # set True after installing NCBI BLAST+ and fetching real sequences
G, homo = build_graph(dis_genes, omim_genes, seqs, use_blast=USE_BLAST)
print(f"graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges, {homo} homology edge(s)")

# fast settings for the notebook; raise to dim=128, walk_len=40, num_walks=10 for the final result
model = node2vec_embed(G, dim=64, walk_len=15, num_walks=3)
disease_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "disease"]
print("trained. embedding dim for Breast:", model.wv[D1].shape)

## Step C — Data Evaluation

In [ ]:
# ---- C1 baselines ; C2 similarity scoring ; C3 calibration & banding ----
def jaccard(a, b):
    A, B = set(a), set(b)
    return len(A & B)/len(A | B) if (A | B) else 0.0

def weighted_cosine(a, b, gene_deg, N):
    keys = set(a) | set(b)
    def vec(d): return {g: d.get(g,0.0)*math.log((N+1)/(gene_deg.get(g,0)+1)) for g in keys}
    va, vb = vec(a), vec(b)
    dot = sum(va[g]*vb[g] for g in keys)
    na = math.sqrt(sum(v*v for v in va.values())); nb = math.sqrt(sum(v*v for v in vb.values()))
    return dot/(na*nb) if na and nb else 0.0

def n2v_cosine(model, n1, n2):
    if n1 not in model.wv or n2 not in model.wv: return None
    v1, v2 = model.wv[n1], model.wv[n2]
    return (float(np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))) + 1.0)/2.0

def calibrate(model, raw, disease_nodes, k=400, seed=1):
    rng = random.Random(seed); sample = []
    for _ in range(k):
        a, b = rng.choice(disease_nodes), rng.choice(disease_nodes)
        if a == b: continue
        s = n2v_cosine(model, a, b)
        if s is not None: sample.append(s)
    return sum(1 for s in sample if s <= raw)/len(sample) if sample else raw

def band(x):
    if x < 0.1:  return "~0   (unrelated / independent)"
    if x < 0.4:  return "~0.2 (not really similar)"
    if x < 0.75: return "~0.6 (somewhat / very similar)"
    return "~1   (highly / maximally similar)"

# ---- C4  Validation against the MeSH disease hierarchy (#16) ----
def pair_shared_categories(n1, n2, did2tree):
    return sorted(did2tree.get(n1, set()) & did2tree.get(n2, set()))

def ontology_auroc(model, disease_nodes, did2tree, k=600, seed=7):
    cands = [d for d in disease_nodes if d in model.wv and d in did2tree]
    if len(cands) < 5: return None, 0, 0
    rng = random.Random(seed); y, s = [], []
    for _ in range(k):
        a, b = rng.choice(cands), rng.choice(cands)
        if a == b: continue
        sc = n2v_cosine(model, a, b)
        if sc is None: continue
        y.append(1 if (did2tree[a] & did2tree[b]) else 0); s.append(sc)
    if len(set(y)) < 2: return None, sum(y), len(y)
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, s), sum(y), len(y)

In [ ]:
# ---- RUN STEP C : score + validate Breast vs Pancreatic ----
jac = jaccard(g1, g2)
wco = weighted_cosine(g1, g2, gene_deg, N)
raw = n2v_cosine(model, D1, D2)
cal = calibrate(model, raw, disease_nodes)

print(f"{id2name[D1]}  vs  {id2name[D2]}\n")
print(f"  [C1 baseline]  Jaccard          = {jac:.3f}")
print(f"  [C1 baseline]  weighted cosine  = {wco:.3f}")
print(f"  [C2 ML]        node2vec (raw)   = {raw:.3f}")
print(f"  [C3 ML]        FINAL score      = {cal:.3f}   ->  {band(cal)}")

# C4 validation using #16 disease vocabulary
sc = pair_shared_categories(D1, D2, did2tree)
print(f"  [C4 validate]  {'share MeSH '+str(sc)+' -> RELATED' if sc else 'no shared MeSH branch -> unrelated'}")
auroc, npos, ntot = ontology_auroc(model, disease_nodes, did2tree)
if auroc is not None:
    print(f"  [C4 corpus]    node2vec vs MeSH AUROC = {auroc:.3f}  ({ntot} pairs, {npos} similar; >0.5 = agrees)")

## Visualizations

In [ ]:
# ---- Figure 1 : shared-gene network ----
sh = sorted(set(g1) & set(g2), key=lambda g: -(g1[g]+g2[g]))[:18]
fig, ax = plt.subplots(figsize=(9, 8)); yc = np.linspace(0.05, 0.95, len(sh))
for y, gene in zip(yc, sh):
    ax.plot([0.12,0.5],[0.5,y], color="#b9c0c9", lw=1); ax.plot([0.5,0.88],[y,0.5], color="#b9c0c9", lw=1)
    ax.scatter(0.5, y, s=420, color="#8ecae6", edgecolor="#2a6f97", zorder=2)
    ax.text(0.5, y, gene, ha="center", va="center", fontsize=7.5, zorder=3)
for x, did, col in [(0.12, D1, "#e07a5f"), (0.88, D2, "#81b29a")]:
    ax.scatter(x, 0.5, s=2600, color=col, edgecolor="black", zorder=4)
    ax.text(x, 0.5, id2name[did].split(",")[0].replace(" ","\n"), ha="center", va="center",
            fontsize=8, weight="bold", zorder=5)
ax.set_title(f"Shared-gene network ({len(set(g1)&set(g2))} shared; top 18 shown)"); ax.axis("off"); plt.show()

In [ ]:
# ---- Figure 2 : method comparison ----
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.bar(0, jac, 0.5, color="#cbd5e1", label="Jaccard (baseline)")
ax.bar(1, wco, 0.5, color="#94a3b8", label="Weighted cosine (baseline)")
ax.bar(2, cal, 0.5, color="#2a6f97", label="node2vec (calibrated)")
ax.set_xticks([0,1,2]); ax.set_xticklabels(["Jaccard","Weighted\ncosine","node2vec"])
ax.set_ylim(0,1); ax.set_ylabel("similarity [0-1]")
ax.set_title("Breast vs Pancreatic: baselines vs node2vec"); ax.legend(); ax.grid(axis="y", alpha=0.3); plt.show()

In [ ]:
# ---- Figure 3 : node2vec embedding map (PCA to 2D) ----
from sklearn.decomposition import PCA
dn = [n for n in disease_nodes if n in model.wv]
sample = random.Random(0).sample(dn, min(400, len(dn)))
for h in (D1, D2):
    if h not in sample: sample.append(h)
xy = PCA(n_components=2).fit_transform(np.array([model.wv[n] for n in sample]))
fig, ax = plt.subplots(figsize=(8.5, 7)); ax.scatter(xy[:,0], xy[:,1], s=12, color="#cbd5e1", alpha=0.6)
for k, (h, col) in enumerate(zip((D1, D2), ["#e07a5f","#81b29a"])):
    i = sample.index(h); dy = 0.10 if k==0 else -0.10; va = "bottom" if k==0 else "top"
    ax.scatter(xy[i,0], xy[i,1], s=120, color=col, edgecolor="black", zorder=3)
    ax.text(xy[i,0], xy[i,1]+dy, id2name[h].split(",")[0], fontsize=9, weight="bold", ha="center", va=va)
ax.set_title("node2vec disease embeddings (PCA to 2D)"); ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.grid(alpha=0.3); plt.show()

In [ ]:
# ---- Figure 4 : calibration ----
rng = random.Random(1)
sims = [s for _ in range(1500) for s in [n2v_cosine(model, rng.choice(dn), rng.choice(dn))] if s is not None]
fig, ax = plt.subplots(figsize=(8.5, 5)); ax.hist(sims, bins=40, color="#cbd5e1", edgecolor="white")
ax.axvline(raw, color="#e07a5f", lw=2.5, label=f"Breast-Pancreatic raw={raw:.2f}\ncalibrated={cal:.2f}")
ax.set_title("Calibration: pair score vs 1500 random disease pairs")
ax.set_xlabel("raw node2vec similarity [0-1]"); ax.set_ylabel("count"); ax.legend(); ax.grid(axis="y", alpha=0.3); plt.show()

### Done — all three CTD files used
- **#7** curated gene-disease → the scores (Steps A–C)
- **#18** gene vocabulary → UniProt column for BLAST (A6)
- **#16** disease vocabulary → C4 validation against the MeSH hierarchy

*For real BLAST:* install NCBI BLAST+, run the fetch cell, set `USE_BLAST = True` in Step B.
*For the final run:* raise node2vec to `dim=128, walk_len=40, num_walks=10`.